[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyneuro/SCP/blob/main/4_synapses.ipynb)

# SCP Step 4 - Synapse Tuning with BMTool

Tune chemical synapse mechanism parameters using BMTool's synaptic tuner with an SCP cell/tune directory.

Workflow:
1. Prepare an SCP cell/tune for BMTool.
2. Define BMTool `general_settings` and `conn_type_settings`.
3. Initialize `SynapseTuner` with the SCP-built cell.
4. Use BMTool's single-event, interactive, frequency-response, and optional optimizer tools.
5. Print a copyable `syns.params` block for SCP synapse group configs.

This notebook intentionally stays close to BMTool's original synaptic tuner notebook. SCP-specific machinery is isolated in `modules.tuning.bmtool_synapse_adapter`.


In [ ]:
# Environment setup: works locally or in Google Colab
%load_ext autoreload
%autoreload 2

import json
import os
import subprocess
import sys
from pathlib import Path

# User-editable only when running in a fresh Colab or unusual local layout.
SCP_REPO_URL = os.environ.get("SCP_REPO_URL", "https://github.com/cyneuro/SCP.git")
SCP_REPO_BRANCH = os.environ.get("SCP_REPO_BRANCH", "") or None
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
AUTO_CLONE_REPO = os.environ.get("SCP_AUTO_CLONE", "1" if IN_COLAB else "0") not in {"0", "false", "False"}
INSTALL_DEPS = None  # None = install automatically in Colab, do not install locally.
SCP_REPO_DIR = Path(os.environ.get("SCP_REPO_DIR", "/content/SCP" if IN_COLAB else str(Path.cwd() / "SCP")))


def _looks_like_scp_repo(path: Path) -> bool:
    return (path / "modules").is_dir() and (path / "run_pipeline.py").is_file()


# Minimal pre-import bootstrap. Fresh Colab cannot import SCP helpers until the repo exists.
repo_root = None
env_root = os.environ.get("SCP_ROOT")
if env_root and _looks_like_scp_repo(Path(env_root).expanduser()):
    repo_root = Path(env_root).expanduser().resolve()
else:
    start = Path.cwd().resolve()
    candidates = list((start, *start.parents))
    for base in (start, start.parent):
        try:
            candidates.extend(child for child in base.iterdir() if child.is_dir())
        except Exception:
            pass
    for candidate in candidates:
        if _looks_like_scp_repo(candidate):
            repo_root = candidate.resolve()
            break

if repo_root is None:
    if not AUTO_CLONE_REPO:
        raise FileNotFoundError("Could not find SCP. Set SCP_ROOT or enable SCP_AUTO_CLONE=1.")
    clone_url = SCP_REPO_URL
    token = os.environ.get("SCP_GIT_TOKEN") or os.environ.get("SCP_GITHUB_TOKEN") or os.environ.get("GITHUB_TOKEN")
    if token and clone_url.startswith("https://") and "@" not in clone_url:
        clone_url = clone_url.replace("https://", f"https://{token}@", 1)
    clone_cmd = ["git", "clone", "--depth", "1"]
    if SCP_REPO_BRANCH:
        clone_cmd += ["--branch", SCP_REPO_BRANCH]
    clone_cmd += [clone_url, str(SCP_REPO_DIR)]
    subprocess.check_call(clone_cmd)
    repo_root = SCP_REPO_DIR.resolve()

os.environ["SCP_ROOT"] = str(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from modules.notebooks.bootstrap import ensure_notebook_dependencies, ensure_python_package, is_colab

ensure_notebook_dependencies(install_deps=INSTALL_DEPS)
if INSTALL_DEPS is True or (INSTALL_DEPS is None and is_colab()):
    ensure_python_package("ipywidgets")
    ensure_python_package("tqdm")

from modules.tuning import ensure_bmtool_on_syspath

bmtool_path = ensure_bmtool_on_syspath(repo_root=repo_root, auto_clone=None)

print("Runtime:", "Colab" if is_colab() else "local")
print("SCP repo:", repo_root)
print("BMTool repo:", bmtool_path)


## 4.1 Prepare SCP Cell

Choose the prepared tune that contains the post-synaptic cell to use for tuning.

Quick guide:
- `cell_name`: folder under `cells/`, for example `PV` or `SST`.
- `tune_name`: tune folder under `cells/<cell>/tunes/`.
- `tune_dir_override`: optional direct path; leave `None` for normal repo layout.
- `RECOMPILE_MODFILES`: set `True` only when mechanism files changed.
- `LOAD_COMPILED_DLL`: normally `True`; set `False` only if another mechanism library is already loaded and you are managing NEURON state manually.


In [ ]:
cell_name = "SST"
tune_name = "seg_tuned"
tunes_parent = "tunes"
tune_dir_override = None

RECOMPILE_MODFILES = False
LOAD_COMPILED_DLL = True

from modules.tuning import prepare_scp_synapse_tuning

session = prepare_scp_synapse_tuning(
    cell_name=cell_name,
    tune_name=tune_name,
    tunes_parent=tunes_parent,
    tune_dir_override=tune_dir_override,
    repo_root=repo_root,
    recompile_modfiles=RECOMPILE_MODFILES,
    load_compiled_dll=LOAD_COMPILED_DLL,
)

cell = session.cell
tune_dir = session.tune_dir

print("Tune directory:", tune_dir)
print("Mechanisms:", json.dumps(session.mechanism_summary, indent=2))
print("Cell:", cell)


## 4.2 Define BMTool Synapse Settings

This section mirrors BMTool's original `general_settings` and `conn_type_settings` structure.

Quick guide:
- `connection`: key in `conn_type_settings` for the synapse type you want to tune.
- `general_settings`: simulation settings used by BMTool's single-event and train protocols.
- `spec_settings.level_of_detail`: NEURON point-process mechanism name from the compiled modfiles.
- `spec_settings.sec_id` / `sec_x`: section index and normalized section location for placing the test synapse.
- `spec_syn_param`: mechanism parameters to initialize before tuning.


In [ ]:
connection = "Fac2SST"  # options below: Fac2SST, Dep2PV, Inh2SST, PV2SST

general_settings = {
    "vclamp": True,
    "rise_interval": (0.1, 0.9),
    "tstart": 500.0,
    "tdur": 100.0,
    "threshold": -15.0,
    "delay": 1.3,
    "weight": 1.0,
    "dt": 0.025,
    "celsius": 20,
}

conn_type_settings = {
    "Fac2SST": {
        "spec_settings": {
            "post_cell": "SCP_Cell",  # not used when `hoc_cell=session.cell` is supplied
            "vclamp_amp": -65.0,
            "sec_x": 0.5,
            "sec_id": 0,
            "level_of_detail": "AMPA_NMDA_STP",
        },
        "spec_syn_param": {
            "initW": 0.25,
            "tau_r_AMPA": 0.2,
            "tau_d_AMPA": 1.7,
            "Use": 0.75,
            "Dep": 0.0,
            "Fac": 200.0,
            "NMDA_ratio": 1.5,
        },
    },
    "Dep2PV": {
        "spec_settings": {
            "post_cell": "SCP_Cell",
            "vclamp_amp": -71.0,
            "sec_x": 0.5,
            "sec_id": 0,
            "level_of_detail": "AMPA_NMDA_STP",
        },
        "spec_syn_param": {
            "initW": 1.5,
            "tau_r_AMPA": 3.5,
            "tau_d_AMPA": 4.0,
            "Use": 0.80,
            "Dep": 100.0,
            "Fac": 0.0,
            "NMDA_ratio": 0.0,
        },
    },
    "Inh2SST": {
        "spec_settings": {
            "post_cell": "SCP_Cell",
            "vclamp_amp": -65.0,
            "sec_x": 0.5,
            "sec_id": 0,
            "level_of_detail": "GABA_A",
        },
        "spec_syn_param": {
            "initW": 0.1,
            "tau_r_GABAA": 0.5,
            "tau_d_GABAA": 5.5,
            "e_GABAA": -75.0,
            "gmax": 0.001,
        },
    },
    "PV2SST": {
        "spec_settings": {
            "post_cell": "SCP_Cell",
            "vclamp_amp": -65.0,
            "sec_x": 0.5,
            "sec_id": 0,
            "level_of_detail": "GABA_A_STP",
        },
        "spec_syn_param": {
            "initW": 5.0,
            "tau_r_GABAA": 0.5,
            "tau_d_GABAA": 5.5,
            "e_GABAA": -75.0,
            "gmax": 0.001,
            "Use": 1.0,
            "Dep": 250.0,
            "Fac": 0.0,
        },
    },
}

print("Selected connection:", connection)
print("Mechanism:", conn_type_settings[connection]["spec_settings"]["level_of_detail"])


## 4.3 Initialize BMTool SynapseTuner

BMTool's original notebook initializes `SynapseTuner` with template and mechanism paths. In SCP, the adapter supplies the already-built cell through `hoc_cell`, while keeping BMTool's tuning API unchanged.

Quick guide:
- Leave `slider_vars = None` to use SCP defaults for the selected mechanism.
- Leave `other_vars_to_record = None` to record common mechanism-specific variables.
- Set either variable explicitly if your custom mechanism exposes different range variables.


In [ ]:
from modules.tuning import (
    create_scp_synapse_tuner,
    default_record_vars_for_connection,
    default_slider_vars_for_connection,
    import_bmtool_synapse_api,
)

SynapseTuner, SynapseOptimizer = import_bmtool_synapse_api(repo_root=repo_root)

connection_settings = conn_type_settings[connection]
slider_vars = None  # e.g., ["initW", "Dep", "Fac", "Use", "tau_r_AMPA", "tau_d_AMPA"]
other_vars_to_record = None  # e.g., ["record_Pr", "record_use"] or ["g"]
current_name = "i"

resolved_slider_vars = slider_vars or default_slider_vars_for_connection(connection_settings)
resolved_record_vars = other_vars_to_record or default_record_vars_for_connection(connection_settings)

tuner = create_scp_synapse_tuner(
    session,
    conn_type_settings=conn_type_settings,
    connection=connection,
    general_settings=general_settings,
    current_name=current_name,
    other_vars_to_record=resolved_record_vars,
    slider_vars=resolved_slider_vars,
)

print("Slider variables:", resolved_slider_vars)
print("Recorded synapse variables:", resolved_record_vars)


## 4.4 Single Event

Run BMTool's single-event protocol. This is usually the first check before opening the interactive tuner.


In [ ]:
tuner.SingleEvent()


## 4.5 Interactive Tuner

Open BMTool's interactive tuner. Use the sliders to inspect how parameter changes alter the single-event and train responses.


In [ ]:
tuner.InteractiveTuner()


## 4.6 Frequency Response

Run BMTool's short-term plasticity frequency-response helper. This reports paired-pulse ratio, induction, and recovery across stimulation frequencies.


In [ ]:
results = tuner.stp_frequency_response(log_plot=False)
results


## 4.7 Optional SynapseOptimizer

BMTool can also optimize selected mechanism parameters toward target metrics. Treat this as optional; manual tuning with the interactive tuner is often easier to reason through first.

Quick guide:
- `param_bounds`: parameters and search bounds for the selected mechanism.
- `target_metrics`: desired BMTool metrics.
- `custom_cost`: controls which target errors matter most.


In [ ]:
# Create the optimizer
optimizer = SynapseOptimizer(tuner)

# Example bounds for AMPA_NMDA_STP. Edit for GABA or custom mechanisms.
param_bounds = {
    "Dep": (0.0, 200.0),
    "Fac": (0.0, 400.0),
    "Use": (0.1, 1.0),
    "tau_r_AMPA": (0.1, 4.0),
    "tau_d_AMPA": (1.0, 20.0),
}
param_bounds = {k: v for k, v in param_bounds.items() if hasattr(tuner.syn, k)}

target_metrics = {
    "induction": -0.75,
    "ppr": 0.8,
    "recovery": 0.0,
    "max_amplitude": 25.0,
    "rise_time": 2.0,
    "decay_time": 9.0,
}


def custom_cost(metrics, targets):
    induction_error = (metrics["induction"] - targets["induction"]) ** 2
    ppr_error = (metrics["ppr"] - targets["ppr"]) ** 2
    recovery_error = (metrics["recovery"] - targets["recovery"]) ** 2
    rise_time_error = (metrics["rise_time"] - targets["rise_time"]) ** 2
    decay_time_error = (metrics["decay_time"] - targets["decay_time"]) ** 2
    return induction_error + 3 * ppr_error + recovery_error + rise_time_error + decay_time_error


if not param_bounds:
    raise ValueError("No optimizer parameters match the selected mechanism. Update param_bounds.")

result = optimizer.optimize_parameters(
    target_metrics=target_metrics,
    param_bounds=param_bounds,
    run_single_event=True,
    run_train_input=True,
    train_frequency=50,
    train_delay=250,
    init_guess="random",  # "random" or "middle_guess"
    cost_function=custom_cost,
    method="SLSQP",
)

optimizer.plot_optimization_results(result)
result


## 4.8 Export Tuned Parameters for SCP

After manual tuning or optimization, print a copyable `syns.params` block. Paste these values into the relevant `cells/<cell>/tunes/<tune>/cell_configs/syn_groups/*.json` file when you are ready to keep the tuned parameters.

This cell does not edit config files automatically.


In [ ]:
from modules.tuning import get_tuned_synapse_params, print_syn_group_param_block

tuned_params = get_tuned_synapse_params(tuner)
print_syn_group_param_block(tuned_params)


## Extra Notes

- Restart the kernel if NEURON reports that a different mechanism library is already loaded.
- If a slider variable is missing, confirm the selected mechanism exposes that parameter in its `.mod` file.
- For custom SCP models, keep the BMTool dictionary structure the same and change only the mechanism name, section location, and parameter names.
